In [1]:
import pandas as pd
import numpy as np
import random

In [193]:
class MyKMeans:
    def __init__(self, n_clusters = 3, max_iter = 10, n_init = 3, random_state = 42):
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.n_init = n_init
        self.random_state = random_state

        self.cluster_centers_ = []
        self.wcss = []
        self.inertia_ = None
        

    def __str__(self):
        return f"MyKMeans class: n_clusters={self.n_clusters}, max_iter={self.max_iter}, " \
               f"n_init={self.n_init}, random_state={self.random_state}"

    def fit(self, X: pd.DataFrame):
        np.random.seed(self.random_state)

        for j in range(self.n_init):

            cluster_centers_ = []
            for i in range(self.n_clusters):
                center = []
                for column in X.columns:
                    min_f = X[column].min()
                    max_f = X[column].max()
                    center.append(np.random.uniform(min_f, max_f))
    
                cluster_centers_.append(center)
            for i in range(self.max_iter):
                cluster_distances = []
    
                for cluster_index, cluster_center in enumerate(cluster_centers_):
                    cluster_center_np = np.array(cluster_center)
                    distances = ((X - cluster_center_np) ** 2).sum(axis=1, skipna=False) ** 0.5
                    distances.name = cluster_index
                    cluster_distances.append(distances)
    
                cluster_distances_df = pd.concat(cluster_distances, axis=1)
                
                y = cluster_distances_df.idxmin(axis=1)
    
                new_cluster_centers = []
                
                for cluster_index in range(self.n_clusters):
                    mask = y == cluster_index
                    X_mask = X[mask]
                    m = len(X_mask)
                    if m > 0:
                        new_cluster_centers.append(X[mask].mean(axis=0).to_list())
                    else:
                        new_cluster_centers.append(cluster_centers_[cluster_index])
                        
                if new_cluster_centers == cluster_centers_:
                    break

                cluster_centers_ = new_cluster_centers

            cluster_distances = []
    
            for cluster_index, cluster_center in enumerate(cluster_centers_):
                cluster_center_np = np.array(cluster_center)
                distances = ((X - cluster_center_np) ** 2).sum(axis=1, skipna=False) ** 0.5
                distances.name = cluster_index
                cluster_distances.append(distances)

            cluster_distances_df = pd.concat(cluster_distances, axis=1)
            y = cluster_distances_df.idxmin(axis=1)
            wcss = 0
                
            for cluster_index in range(self.n_clusters):
                mask = y == cluster_index
                X_mask = X[mask]
                cluster_center_np = np.array(cluster_centers_[cluster_index])
                wcss += ((X_mask - cluster_center_np) ** 2).sum(skipna=False).sum(skipna=False)
            self.wcss.append(wcss)
            self.cluster_centers_.append(cluster_centers_)
            
        self.cluster_centers_, self.inertia_ = min(zip(self.cluster_centers_, self.wcss), key= lambda x: x[1])

        self.cluster_centers_ = [cluster_center for cluster_center in self.cluster_centers_ if not any(np.isnan(cluster_center))]
        
    def predict(self, X: pd.DataFrame):
        cluster_distances = []
        for cluster_index, cluster_center in enumerate(self.cluster_centers_):
            cluster_center_np = np.array(cluster_center)
            distances = ((X - cluster_center_np) ** 2).sum(axis=1, skipna=False) ** 0.5
            distances.name = cluster_index
            cluster_distances.append(distances)

        cluster_distances_df = pd.concat(cluster_distances, axis=1)
        y = cluster_distances_df.idxmin(axis=1)
            
        return y

In [194]:
X = pd.DataFrame({'a':range(10), 'b': range(10, 0, -1)})

In [195]:
pd.DataFrame.sum()

TypeError: DataFrame.sum() missing 1 required positional argument: 'self'

In [196]:
(X - np.array([np.NaN, np.NaN]) ** 2).sum(axis=1, skipna=False)

0   NaN
1   NaN
2   NaN
3   NaN
4   NaN
5   NaN
6   NaN
7   NaN
8   NaN
9   NaN
dtype: float64

In [197]:
my_k_means = MyKMeans()

In [198]:
my_k_means.fit(X)
my_k_means.predict(X)

0    2
1    2
2    0
3    0
4    0
5    0
6    1
7    1
8    1
9    1
dtype: int64

In [189]:
import matplotlib.pyplot as plt 

In [162]:
any(np.isnan(np.array([np.NaN])))

True